# DeepLab v3+（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
DeepLab v3+は，DeepLab v3で導入されたASPP（Atrous Spatial Pyramid Pooling）による広い受容野のマルチスケール文脈情報に加え，バックボーンの浅い層が持つ高解像度な特徴を融合する**学習可能なデコーダモジュール**を追加した手法である．本ノートブックでは，dilated convolution化したResNet50バックボーンとASPPをDeepLab v3から引き継ぎつつ，デコーダを新たに実装し，PASCAL VOC 2007データセットを用いてセマンティックセグメンテーションを行う．DeepLab v3にはないデコーダの役割（低レベル特徴との融合による境界の精緻化）を理解することを目標とする．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCSegmentation
from torchvision.models import resnet50, ResNet50_Weights
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して画素単位のクラスラベル（セグメンテーションマスク）が付与されたデータセットで，学習用（trainval）422枚，評価用（test）210枚の画像から構成されます（矩形領域のみのVOCDetectionと異なり，セグメンテーション用のアノテーションが存在する画像のみに限られるため，画像分類・物体検出のノートブックと比べて画像枚数が少なくなっています）．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．マスク画像は，各画素の値がクラスID（0が背景，1〜20が物体クラス，255が物体の境界など評価から除外する「無視領域」）になっている画像として与えられます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します．ここでは，VOCdevkitで標準的に使われているビット演算によるカラーパレットの生成方法を使用します．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## Dilated ResNet50バックボーン
DeepLab v3+のバックボーンは，DeepLab v3と全く同じ，事前学習済みResNet50をdilated convolution（atrous convolution）化したものを使用する．通常のResNet50は`conv1`から`layer4`まで通すと出力ストライド（入力に対する縮小率）が32になるが，これでは特徴マップの解像度が低くなりすぎ，物体境界などの細かい構造を復元しにくくなる．そこで`layer3`・`layer4`の最初のブロックにおいて，ストライドによるダウンサンプリングを行わない代わりに畳み込みのdilation（隙間）を広げることで，受容野を保ったまま出力ストライドを8に抑える（`layer3`はdilation=2，`layer4`はdilation=4）．畳み込みのカーネルサイズ自体は変化しないため，事前学習済みの重みをそのまま利用できる．

DeepLab v3+ではさらに，デコーダで利用する`layer1`の出力（stride 4の高解像度な低レベル特徴）も別途取り出せるようにしておく．

In [ ]:
def dilate_layer(layer, dilation):
    """ResNetの1ステージ（layer3やlayer4のnn.Sequential）を，
    出力ストライドを増やさずに受容野を広げるdilated convolutionへ書き換えるヘルパー関数．
    最初のBottleneckブロックのみstrideを1にしてダウンサンプリングを無効化し，
    ステージ内の全ブロックの3x3畳み込みのdilationとpaddingをdilationに設定する．
    畳み込みの重み形状（カーネルサイズ）は変化しないため，事前学習済み重みはそのまま流用できる．
    """
    first_block = layer[0]
    first_block.conv2.stride = (1, 1)
    if first_block.downsample is not None:
        first_block.downsample[0].stride = (1, 1)
    for block in layer:
        block.conv2.dilation = (dilation, dilation)
        block.conv2.padding = (dilation, dilation)
    return layer


class DilatedResNet50Backbone(nn.Module):
    """事前学習済みResNet50をベースに，layer3・layer4をdilated convolution化したバックボーン．
    ASPPへの入力となる出力ストライド8の深い特徴と，デコーダへの入力となるストライド4の低レベル特徴の両方を返す．
    """
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)  # stride 4
        self.layer1 = resnet.layer1  # stride 4を維持, channels 256（デコーダで使う低レベル特徴）
        self.layer2 = resnet.layer2  # stride 4 -> 8, channels 512
        self.layer3 = dilate_layer(resnet.layer3, dilation=2)  # stride 8を維持, channels 1024
        self.layer4 = dilate_layer(resnet.layer4, dilation=4)  # stride 8を維持, channels 2048

    def forward(self, x):
        x = self.stem(x)
        low_level_feat = self.layer1(x)  # stride 4
        x = self.layer2(low_level_feat)  # stride 8
        x = self.layer3(x)               # stride 8 (dilation=2)
        x = self.layer4(x)               # stride 8 (dilation=4)
        return x, low_level_feat


# 出力ストライドの確認：入力256x256に対し，深い特徴は32x32（1/8），低レベル特徴は64x64（1/4）になっているはず
_backbone_check = DilatedResNet50Backbone().to(device)
with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    _feat, _low = _backbone_check(_dummy)
print('deep feature (ASPPへの入力):', _feat.shape)
print('low-level feature (デコーダへの入力):', _low.shape)
del _backbone_check, _dummy, _feat, _low

## ASPP（Atrous Spatial Pyramid Pooling）
ASPPは，DeepLab v3と全く同じモジュールである．バックボーンの出力（stride 8）に対して，異なるdilation rate（1, 6, 12, 18）を持つ並列な畳み込みブランチと，特徴マップ全体を平均プーリングしてから1x1畳み込み・アップサンプリングするグローバルブランチを適用し，それらをチャンネル方向にconcatしてから1x1畳み込みで統合する．PSPNetのPyramid Pooling Moduleが複数スケールの**プーリング**でマルチスケール文脈を得るのに対し，ASPPは複数の**dilation rate**を持つ畳み込みでマルチスケール文脈を得る点が異なる．

In [ ]:
class ASPP(nn.Module):
    """異なるdilation rateの畳み込みとグローバルプーリングを並列に適用するモジュール"""
    def __init__(self, in_channels=2048, out_channels=256, rates=(6, 12, 18)):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=rates[0], dilation=rates[0], bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=rates[1], dilation=rates[1], bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.branch4 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=rates[2], dilation=rates[2], bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.global_branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Dropout2d(0.5),
        )

    def forward(self, x):
        size = x.shape[2:]
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        b5 = self.global_branch(x)
        b5 = F.interpolate(b5, size=size, mode='bilinear', align_corners=False)  # 1x1のグローバル特徴を元の解像度に戻す
        out = torch.cat([b1, b2, b3, b4, b5], dim=1)
        return self.project(out)

## デコーダモジュール（DeepLab v3からの最大の変更点）
DeepLab v3は，ASPPの出力（stride 8）を1x1畳み込みでクラス数チャンネルに変換したのち，入力サイズまで一気にバイリニア補間でアップサンプリングするだけであった（低レベル特徴との融合は行わない）．8倍もの一気のアップサンプリングでは，物体の輪郭などの細かい情報が失われたままになってしまう．

DeepLab v3+では，この点を改善するために**学習可能なデコーダモジュール**を追加する．具体的には，

1. バックボーンの浅い層（`layer1`の出力，stride 4）を1x1畳み込みで少ないチャンネル数（48）に削減する（チャンネル数を絞るのは，情報量の多いASPP出力に対して低レベル特徴の影響が大きくなりすぎないようにするためである）．
2. ASPPの出力（stride 8）をstride 4までバイリニア補間でアップサンプリングし，1.の低レベル特徴とチャンネル方向にconcatする．
3. 3x3畳み込みを2層適用して特徴を精緻化したのち，1x1畳み込みでクラス数チャンネルに変換する．
4. 最後に入力サイズ（stride 1）までバイリニア補間でアップサンプリングする．

このように，ASPP出力からのアップサンプリングを「stride 8 → 4（デコーダで低レベル特徴と融合）→ 1（最終アップサンプリング）」の2段階に分け，浅い層の高解像度特徴を融合する点が，低レベル特徴を一切使わないDeepLab v3にはない，DeepLab v3+最大の特徴である．

In [ ]:
class Decoder(nn.Module):
    """ASPP出力（低解像度）と，バックボーンの浅い層の出力（高解像度）を融合するデコーダ"""
    def __init__(self, low_level_channels=256, low_level_out=48, aspp_channels=256, n_class=n_class):
        super().__init__()
        self.reduce_low_level = nn.Sequential(
            nn.Conv2d(low_level_channels, low_level_out, kernel_size=1, bias=False),
            nn.BatchNorm2d(low_level_out), nn.ReLU(inplace=True),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(aspp_channels + low_level_out, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(256, n_class, kernel_size=1)

    def forward(self, aspp_out, low_level_feat):
        low = self.reduce_low_level(low_level_feat)
        aspp_up = F.interpolate(aspp_out, size=low.shape[2:], mode='bilinear', align_corners=False)  # stride 8 -> 4
        x = torch.cat([aspp_up, low], dim=1)
        x = self.refine(x)
        return self.classifier(x)

## DeepLab v3+の全体構成
バックボーン・ASPP・デコーダを組み合わせて，DeepLab v3+全体を構築する．デコーダの出力（stride 4）は，最後に入力画像と同じ解像度までバイリニア補間でアップサンプリングする．

In [ ]:
class DeepLabV3Plus(nn.Module):
    def __init__(self, n_class=n_class):
        super().__init__()
        self.backbone = DilatedResNet50Backbone()
        self.aspp = ASPP(in_channels=2048, out_channels=256)
        self.decoder = Decoder(low_level_channels=256, low_level_out=48, aspp_channels=256, n_class=n_class)

    def forward(self, x):
        input_size = x.shape[2:]
        feat, low_level_feat = self.backbone(x)
        aspp_out = self.aspp(feat)
        out = self.decoder(aspp_out, low_level_feat)
        out = F.interpolate(out, size=input_size, mode='bilinear', align_corners=False)  # stride 4 -> 1
        return out

## 損失関数
セグメンテーションは，画素ごとのクラス分類問題とみなせるため，`nn.CrossEntropyLoss`をそのまま利用できる．マスクに含まれる無視領域（ラベル255）は`ignore_index`で損失計算から除外する．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
DeepLab v3+を構築し，最適化手法としてAdamを設定する．バックボーンには事前学習済みの重みを使用しているため，FCNなどと同様に小さい学習率を用いる．

In [ ]:
model = DeepLabV3Plus(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてDeepLab v3+を学習する．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定している．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoU（Intersection over Union）の平均であるmIoU（mean IoU）を求める．mIoUの算出には，`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外する．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## セグメンテーション結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示する．

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのDeepLab v3+との違い

| 項目 | オリジナル論文 (Chen et al., 2018) | 本ノートブック |
| :-- | :-- | :-- |
| バックボーン | 改良Xception（Aligned Xception）を主に使用（ResNet-101も比較実験あり） | 事前学習済みResNet50をdilated convolution化したもの |
| ASPP・デコーダの畳み込み | 計算量削減のため，depthwise separable convolutionを使用 | 通常の畳み込み（separableではない）で実装 |
| output stride | 学習時は16，推論時は8に切り替えて使用 | 学習・推論とも一貫してoutput stride=8 |
| 後処理 | なし（DeepLab v1・v2で使われたDenseCRFはv3+では不使用） | なし |
| データ拡張 | ランダムスケール・ランダムクロップなど複数の工夫を実施 | 固定サイズへのリサイズのみ（データ拡張なし） |
| 学習データ量 | PASCAL VOCの拡張データ（SBD, JFT-300Mなど）を含む大規模データ | VOC2007のtrainvalのみ（422枚） |

## 課題

1. `epoch_num`や学習率を変更し，mIoUがどのように変化するか確認してください．
2. デコーダを使わず，ASPPの出力を直接入力サイズまでアップサンプリングするDeepLab v3相当のモデルを実装し，本ノートブックのDeepLab v3+と比較してください．低解像度特徴と高解像度特徴を融合するデコーダの有無で，物体境界付近の予測がどのように変化するか観察してください．
3. デコーダの低レベル特徴のチャンネル数（`low_level_out=48`）を変更し，mIoUや予測結果がどのように変化するか確認してください．